In [1]:
#0
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/KAZ_MORPH"
SRC  = f"{BASE}/merge_seg_wordforms.txt"

OUT_DIR   = f"{BASE}/femseg_v2_100k"
SAMPLE_TXT= f"{OUT_DIR}/wordforms_100k.txt"
CHAR2ID   = f"{OUT_DIR}/char2id_v2_100k.json"
TRAIN_JSONL = f"{OUT_DIR}/train.jsonl"
VAL_JSONL   = f"{OUT_DIR}/val.jsonl"

import os
os.makedirs(OUT_DIR, exist_ok=True)
print("SRC:", SRC)
print("OUT_DIR:", OUT_DIR)


Mounted at /content/drive
SRC: /content/drive/MyDrive/KAZ_MORPH/merge_seg_wordforms.txt
OUT_DIR: /content/drive/MyDrive/KAZ_MORPH/femseg_v2_100k


In [2]:
#1
import random

K = 100_000
random.seed(42)

sample = []
n = 0

with open(SRC, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        n += 1
        if len(sample) < K:
            sample.append(line)
        else:
            j = random.randint(1, n)
            if j <= K:
                sample[j-1] = line

random.shuffle(sample)

with open(SAMPLE_TXT, "w", encoding="utf-8") as w:
    for s in sample:
        w.write(s + "\n")

print("Saved:", SAMPLE_TXT)
print("Lines:", len(sample), "from total seen:", n)


Saved: /content/drive/MyDrive/KAZ_MORPH/femseg_v2_100k/wordforms_100k.txt
Lines: 100000 from total seen: 2328928


In [3]:
#2
import re

def parse_cse_word(line: str):
    # пример строки: "арттыр@@ а@@ ды"
    # разделитель в ваших данных чаще всего "@@ " (двойной @ + пробел)
    parts = [p.strip() for p in line.split("@@") if p.strip()]
    # Иногда после @@ нет пробела — это тоже ок, split("@@") ловит оба варианта.
    # Убираем пробелы внутри морфов (на всякий случай)
    parts = [re.sub(r"\s+", "", p) for p in parts if p]

    # если вдруг строка без @@ — считаем одним сегментом
    if len(parts) == 0:
        s = re.sub(r"\s+", "", line)
        parts = [s] if s else []

    raw = "".join(parts)
    chars = list(raw)

    labels = []
    for seg in parts:
        seg_chars = list(seg)
        L = len(seg_chars)
        if L == 0:
            continue
        if L == 1:
            labels.append("S")
        else:
            labels.append("B")
            if L > 2:
                labels.extend(["M"]*(L-2))
            labels.append("E")

    # sanity
    if len(chars) != len(labels):
        return None

    return chars, labels

# быстрый тест на одной строке
test_line = "арттыр@@ а@@ ды"
print(test_line, "=>", parse_cse_word(test_line))


арттыр@@ а@@ ды => (['а', 'р', 'т', 'т', 'ы', 'р', 'а', 'д', 'ы'], ['B', 'M', 'M', 'M', 'M', 'E', 'S', 'B', 'E'])


In [4]:
#3
import json
from collections import Counter

def build_char_vocab_and_jsonl(sample_path, train_jsonl, val_jsonl, char2id_path, val_ratio=0.10):
    # собираем char vocab и одновременно пишем jsonl (stream)
    # Чтобы не держать все записи в памяти, сначала пройдём файл и соберём вал-индексы.
    # Для 100k это нормально.
    with open(sample_path, "r", encoding="utf-8") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    N = len(lines)
    idx = list(range(N))
    random.seed(42)
    random.shuffle(idx)
    val_n = int(N * val_ratio)
    val_set = set(idx[:val_n])

    char_counter = Counter()

    # 1) собираем char vocab
    parsed = [None]*N
    bad = 0
    for i, line in enumerate(lines):
        out = parse_cse_word(line)
        if out is None:
            bad += 1
            continue
        chars, labels = out
        parsed[i] = (chars, labels)
        char_counter.update(chars)

    # 2) char2id
    # спец. токены
    char2id = {"<pad>": 0, "<unk>": 1}
    for ch, _ in char_counter.most_common():
        if ch not in char2id:
            char2id[ch] = len(char2id)

    with open(char2id_path, "w", encoding="utf-8") as w:
        json.dump(char2id, w, ensure_ascii=False, indent=2)

    # 3) пишем jsonl
    def write_jsonl(path, indices):
        c = 0
        with open(path, "w", encoding="utf-8") as w:
            for i in indices:
                if parsed[i] is None:
                    continue
                chars, labels = parsed[i]
                rec = {"chars": chars, "labels": labels}
                w.write(json.dumps(rec, ensure_ascii=False) + "\n")
                c += 1
        return c

    train_idx = [i for i in range(N) if i not in val_set]
    val_idx   = [i for i in range(N) if i in val_set]

    train_count = write_jsonl(train_jsonl, train_idx)
    val_count   = write_jsonl(val_jsonl, val_idx)

    print("Total lines:", N)
    print("Bad skipped:", bad)
    print("Char vocab:", len(char2id))
    print("Train records:", train_count)
    print("Val records:", val_count)
    print("Saved:", char2id_path, train_jsonl, val_jsonl)

build_char_vocab_and_jsonl(SAMPLE_TXT, TRAIN_JSONL, VAL_JSONL, CHAR2ID, val_ratio=0.10)


Total lines: 100000
Bad skipped: 0
Char vocab: 32
Train records: 90000
Val records: 10000
Saved: /content/drive/MyDrive/KAZ_MORPH/femseg_v2_100k/char2id_v2_100k.json /content/drive/MyDrive/KAZ_MORPH/femseg_v2_100k/train.jsonl /content/drive/MyDrive/KAZ_MORPH/femseg_v2_100k/val.jsonl


In [5]:
#4
!pip -q install pytorch-crf

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))


torch: 2.9.0+cpu
cuda available: False
device: cpu


In [6]:
#5
import json, math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchcrf import CRF

TAG2ID = {"B":0, "M":1, "E":2, "S":3}
ID2TAG = {v:k for k,v in TAG2ID.items()}

with open(CHAR2ID, "r", encoding="utf-8") as f:
    char2id = json.load(f)

PAD_ID = char2id["<pad>"]
UNK_ID = char2id["<unk>"]

class JsonlSeqDataset(Dataset):
    def __init__(self, path):
        self.items = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                self.items.append(rec)

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        rec = self.items[idx]
        char_ids = [char2id.get(ch, UNK_ID) for ch in rec["chars"]]
        tag_ids  = [TAG2ID[t] for t in rec["labels"]]
        return torch.tensor(char_ids, dtype=torch.long), torch.tensor(tag_ids, dtype=torch.long)

def collate_batch(batch):
    # batch: list of (char_ids, tag_ids)
    lens = [len(x[0]) for x in batch]
    max_len = max(lens)

    x = torch.full((len(batch), max_len), PAD_ID, dtype=torch.long)
    y = torch.zeros((len(batch), max_len), dtype=torch.long)
    mask = torch.zeros((len(batch), max_len), dtype=torch.bool)

    for i,(chars,tags) in enumerate(batch):
        L = len(chars)
        x[i,:L] = chars
        y[i,:L] = tags
        mask[i,:L] = True

    return x, y, mask

class FEMSegV2(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, lstm_hidden=256, dropout=0.1, num_tags=4):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_hidden*2, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, x, mask, tags=None):
        # x: [B,T]
        h = self.emb(x)                       # [B,T,emb]
        h,_ = self.lstm(h)                    # [B,T,2H]
        h = self.drop(h)
        emissions = self.fc(h)                # [B,T,num_tags]

        if tags is not None:
            # CRF returns log-likelihood; loss = -loglik
            ll = self.crf(emissions, tags, mask=mask, reduction="mean")
            return -ll
        else:
            pred = self.crf.decode(emissions, mask=mask)
            return pred

train_ds = JsonlSeqDataset(TRAIN_JSONL)
val_ds   = JsonlSeqDataset(VAL_JSONL)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, collate_fn=collate_batch, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False, collate_fn=collate_batch, num_workers=0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FEMSegV2(vocab_size=len(char2id)).to(device)
optim = torch.optim.AdamW(model.parameters(), lr=2e-3)


In [7]:
#6
import time
from pathlib import Path

SAVE_DIR = f"{OUT_DIR}/models"
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
BEST_PATH = f"{SAVE_DIR}/femseg_v2_100k_best.pt"

def token_acc_on_loader(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x,y,mask in loader:
            x,y,mask = x.to(device), y.to(device), mask.to(device)
            pred = model(x, mask, tags=None)  # list[list[int]]
            # сравним по маске
            for i in range(len(pred)):
                L = int(mask[i].sum().item())
                p = pred[i]
                gold = y[i,:L].tolist()
                correct += sum(int(pi==gi) for pi,gi in zip(p, gold))
                total += L
    return correct / max(1,total)

best = 0.0

for epoch in range(1, 4):
    t0 = time.time()
    model.train()
    loss_sum = 0.0
    steps = 0

    for x,y,mask in train_loader:
        x,y,mask = x.to(device), y.to(device), mask.to(device)
        optim.zero_grad()
        loss = model(x, mask, tags=y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()

        loss_sum += loss.item()
        steps += 1

    train_loss = loss_sum / max(1,steps)
    val_acc = token_acc_on_loader(model, val_loader)

    print(f"[Epoch {epoch}] train_loss={train_loss:.4f}  val_token_acc={val_acc:.4f}  time={(time.time()-t0):.1f}s")

    ckpt = f"{SAVE_DIR}/femseg_v2_100k_epoch{epoch}.pt"
    torch.save({"state_dict": model.state_dict(), "char2id": char2id}, ckpt)

    if val_acc > best:
        best = val_acc
        torch.save({"state_dict": model.state_dict(), "char2id": char2id}, BEST_PATH)
        print("[best] updated:", BEST_PATH)

print("Done. Best val_token_acc =", best)


[Epoch 1] train_loss=0.3061  val_token_acc=0.9984  time=242.2s
[best] updated: /content/drive/MyDrive/KAZ_MORPH/femseg_v2_100k/models/femseg_v2_100k_best.pt
[Epoch 2] train_loss=0.0302  val_token_acc=0.9985  time=239.0s
[best] updated: /content/drive/MyDrive/KAZ_MORPH/femseg_v2_100k/models/femseg_v2_100k_best.pt
[Epoch 3] train_loss=0.0247  val_token_acc=0.9983  time=239.2s
Done. Best val_token_acc = 0.99849327217417


In [8]:
#7
def bmes_to_segments(chars, tags):
    segs = []
    cur = []
    for ch, t in zip(chars, tags):
        if t == "S":
            if cur:  # закрыть если было что-то
                segs.append("".join(cur)); cur=[]
            segs.append(ch)
        elif t == "B":
            if cur:
                segs.append("".join(cur))
            cur = [ch]
        elif t == "M":
            cur.append(ch)
        elif t == "E":
            cur.append(ch)
            segs.append("".join(cur))
            cur = []
        else:
            cur.append(ch)
    if cur:
        segs.append("".join(cur))
    return segs

# загрузить best
ckpt = torch.load(BEST_PATH, map_location=device)
model.load_state_dict(ckpt["state_dict"])
model.eval()

def segment_word(raw_word: str):
    raw_word = re.sub(r"\s+", "", raw_word)
    chars = list(raw_word)
    x = torch.tensor([[char2id.get(ch, UNK_ID) for ch in chars]], dtype=torch.long).to(device)
    mask = torch.ones_like(x, dtype=torch.bool).to(device)
    pred = model(x, mask, tags=None)[0]
    tags = [ID2TAG[i] for i in pred]
    segs = bmes_to_segments(chars, tags)
    return segs

print(segment_word("ылғалдандырады"))
print("@@ format:", "@@ ".join(segment_word("ылғалдандырады")))


['ыл', 'ғал', 'дан', 'дыр', 'а', 'ды']
@@ format: ыл@@ ғал@@ дан@@ дыр@@ а@@ ды


In [ ]:
#8
import re, math
import torch

# ==== ВАЖНО: УКАЖИТЕ ПУТИ ====
RAW_TXT     = "/content/drive/MyDrive/KAZ_MORPH/chunk_099_10.txt"          # исходные 10 предложений (raw)
GOLD_SP_TXT = "/content/drive/MyDrive/KAZ_MORPH/chunk_099_10_sp_gold.txt"  # ваш gold SP (создайте/укажите)

# ==== ЗАГРУЗИТЬ BEST v2_100k (как в обучении) ====
BEST_PATH = "/content/drive/MyDrive/KAZ_MORPH/femseg_v2_100k/models/femseg_v2_100k_best.pt"

ckpt = torch.load(BEST_PATH, map_location=device)
model.load_state_dict(ckpt["state_dict"])
model.eval()

# -----------------------------
# 1) Утилиты: SP <-> raw
# -----------------------------
def sp_to_raw(sp_line: str) -> str:
    # ▁Тері ні ▁терең -> Терінітерең... (с пробелами можно иначе, но для границ нам важны символы)
    toks = sp_line.strip().split()
    out = []
    for t in toks:
        if t.startswith("▁"):
            out.append(" " + t[1:])  # слово началось
        else:
            out.append(t)
    return "".join(out).strip()

def sp_boundaries(sp_line: str):
    """
    Возвращает множество позиций-границ между символами (0..len(raw)-2),
    где стоит граница морфемы/токена в SP-строке.
    """
    toks = sp_line.strip().split()
    raw = ""
    bounds = set()
    pos = 0  # текущая позиция в raw (по символам)
    first = True

    for tok in toks:
        if tok.startswith("▁"):
            piece = tok[1:]
            if not first:
                # граница слова (между символами предыдущего и пробелом) — мы пробелы не учитываем,
                # поэтому просто считаем границу перед первым символом нового слова
                # (для морф-граней часто слова не учитывают; но вы считали по сырой строке без пробелов)
                pass
            first = False
        else:
            piece = tok

        # граница перед этим piece (если raw уже не пустой)
        if pos > 0:
            bounds.add(pos-1)  # граница между pos-1 и pos

        raw += piece
        pos += len(piece)

    return raw, bounds

# -----------------------------
# 2) FEMSeg v2 сегментация предложения -> SP
# -----------------------------
ID2TAG = {0:"B", 1:"M", 2:"E", 3:"S"}

def bmes_to_segments(chars, tags):
    segs, cur = [], []
    for ch, t in zip(chars, tags):
        if t == "S":
            if cur:
                segs.append("".join(cur)); cur=[]
            segs.append(ch)
        elif t == "B":
            if cur:
                segs.append("".join(cur))
            cur = [ch]
        elif t == "M":
            cur.append(ch)
        elif t == "E":
            cur.append(ch); segs.append("".join(cur)); cur=[]
        else:
            cur.append(ch)
    if cur:
        segs.append("".join(cur))
    return segs

def segment_word_v2(word: str):
    word = re.sub(r"\s+", "", word)
    if not word:
        return []
    chars = list(word)
    x = torch.tensor([[char2id.get(ch, UNK_ID) for ch in chars]], dtype=torch.long).to(device)
    mask = torch.ones_like(x, dtype=torch.bool).to(device)
    pred = model(x, mask, tags=None)[0]
    tags = [ID2TAG[i] for i in pred]
    return bmes_to_segments(chars, tags)

def femseg_sentence_to_sp(line: str):
    # простая токенизация по словам/знакам
    tokens = re.findall(r"\w+|[^\w\s]", line, flags=re.UNICODE)
    sp_out = []
    for tok in tokens:
        if re.match(r"^\w+$", tok, flags=re.UNICODE):
            segs = segment_word_v2(tok)
            if not segs:
                continue
            sp_out.append("▁" + segs[0])
            sp_out.extend(segs[1:])
        else:
            # пунктуация отдельным токеном (как у вас)
            sp_out.append(tok)
    return " ".join(sp_out)

# -----------------------------
# 3) Edit distance (свой, без Levenshtein)
# -----------------------------
def edit_distance(a: str, b: str) -> int:
    # DP O(n*m) — для 10 строк нормально
    n, m = len(a), len(b)
    if n == 0: return m
    if m == 0: return n
    prev = list(range(m+1))
    for i in range(1, n+1):
        cur = [i] + [0]*m
        ca = a[i-1]
        for j in range(1, m+1):
            cb = b[j-1]
            cost = 0 if ca == cb else 1
            cur[j] = min(
                prev[j] + 1,      # del
                cur[j-1] + 1,     # ins
                prev[j-1] + cost  # sub
            )
        prev = cur
    return prev[m]

# -----------------------------
# 4) Подсчёт Boundary metrics
# -----------------------------
def boundary_metrics(pred_sp_lines, gold_sp_lines):
    TP=FP=FN=TN=0
    used = 0
    total_positions = 0

    for p_line, g_line in zip(pred_sp_lines, gold_sp_lines):
        p_raw, p_b = sp_boundaries(p_line)
        g_raw, g_b = sp_boundaries(g_line)

        # сравниваем на общем raw: если разные — берём минимальный префикс
        L = min(len(p_raw), len(g_raw))
        if L < 2:
            continue

        used += 1
        total_positions += (L-1)

        for i in range(L-1):
            p_is = (i in p_b)
            g_is = (i in g_b)
            if p_is and g_is: TP += 1
            elif p_is and (not g_is): FP += 1
            elif (not p_is) and g_is: FN += 1
            else: TN += 1

    acc = (TP+TN) / max(1, TP+TN+FP+FN)
    prec = TP / max(1, TP+FP)
    rec  = TP / max(1, TP+FN)
    f1   = 2*prec*rec / max(1e-9, (prec+rec))

    return {
        "Sentences used": used,
        "Total positions": total_positions,
        "TP": TP, "FP": FP, "FN": FN, "TN": TN,
        "Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1
    }

# -----------------------------
# 5) Запуск оценки
# -----------------------------
with open(RAW_TXT, "r", encoding="utf-8") as f:
    raw_lines = [ln.rstrip("\n") for ln in f][:10]

with open(GOLD_SP_TXT, "r", encoding="utf-8") as f:
    gold_lines = [ln.rstrip("\n") for ln in f][:10]

pred_lines = [femseg_sentence_to_sp(ln) for ln in raw_lines]

# Average edit distance по raw (из SP восстановим raw и сравним)
dists = []
for p, g in zip(pred_lines, gold_lines):
    pr = sp_to_raw(p).replace(" ", "")
    gr = sp_to_raw(g).replace(" ", "")
    d = edit_distance(pr, gr) / max(1, len(gr))
    dists.append(d)

print("Average edit distance (norm):", sum(dists)/max(1,len(dists)))

m = boundary_metrics(pred_lines, gold_lines)
print("\n=== Boundary metrics (SP-format) FEMSeg_v2_100k ===")
print(f"Sentences used:       {m['Sentences used']}")
print(f"Total positions:      {m['Total positions']}")
print(f"TP (correct boundary): {m['TP']}")
print(f"FP (extra boundary):   {m['FP']}")
print(f"FN (missed boundary):  {m['FN']}")
print(f"TN (correct non-bound):{m['TN']}")
print()
print(f"Boundary Accuracy:  {m['Accuracy']:.4f}")
print(f"Boundary Precision: {m['Precision']:.4f}")
print(f"Boundary Recall:    {m['Recall']:.4f}")
print(f"Boundary F1:        {m['F1']:.4f}")

print("\n--- Example (1st line) ---")
print("RAW:", raw_lines[0])
print("PRED:", pred_lines[0])
print("GOLD:", gold_lines[0])
